<a href="https://colab.research.google.com/github/petersonrs/bibio/blob/main/An%C3%A1lise_Nivel_Rios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nível dos RIOS

In [30]:
PAGE = 'https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios'
path = '/content/drive/MyDrive/Python/Analise Volume Rio/'

# Arquivo para guardar os dados copiados em cache
filename = 'dados_copiados_v1.pickle'

In [31]:
!pip install selenium>=4.20.0

# Instalação do webdriver_manager para gerenciar o chromedriver (se já não estiver instalado)
!pip install webdriver_manager

# Instala o Chrome no ambiente Colab, essencial para o Selenium com ChromeDriver
!apt-get update
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable



Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Get:11 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,298 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,545 kB]
Fetched 2,973 kB in 3s (1,073 kB/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy tru

In [32]:
import re
import sys
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo # Importa ZoneInfo para lidar com fusos horários

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

# ------------------------- CONFIGURAÇÃO -------------------------

URL = "https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios"

# Códigos de estação que você quer salvar no histórico.
# Ex.: ["DC06"] só a estação pedida, ou ["DC06", "DC10"] para mais de uma.
ESTACOES_ALVO = ["DC03","DC04","DC05","DC06"]

# Onde o histórico será salvo (um arquivo parquet por estação)
OUTPUT_DIR = Path("/content/drive/MyDrive/Python/Analise Volume Rio")

# Tempo máximo de espera pelo carregamento do conteúdo dinâmico (segundos)
TIMEOUT_CARREGAMENTO = 25

# Regex que varre o texto "Comparativo" inteiro e extrai cada leitura:
# grupo 1 = código da estação (ex: DC06)
# grupo 2 = descrição da estação
# grupo 3 = data (DD/MM/AAAA)
# grupo 4 = hora (HH:MM:SS)
# grupo 5 = nível do rio (com vírgula decimal, ou número inteiro sem vírgula)
REGEX_LEITURA = re.compile(
    r"(DC\d{2})\s+(.*?)\s+"
    r"(\d{2}/\d{2}/\d{4}),\s*(\d{2}:\d{2}:\d{2})\s+"
    r"(\d+(?:,\d+)?)"
    r"(?=\s+DC\d{2}|\s*$)"
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# Define o fuso horário de São Paulo (que geralmente corresponde ao horário de Brasília)
TIMEZONE_BR = ZoneInfo("America/Sao_Paulo")

# ------------------------- FUNÇÕES -------------------------

def criar_driver() -> webdriver.Chrome:
    """Cria uma instância do Chrome em modo headless."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)


def esperar_conteudo_carregar(driver):
    """Espera o JS da página carregar os dados das estações antes de continuar."""
    wait = WebDriverWait(driver, TIMEOUT_CARREGAMENTO)

    def conteudo_carregado(d):
        body_text = d.find_element(By.TAG_NAME, "body").text
        return any(codigo in body_text for codigo in ESTACOES_ALVO)

    try:
        wait.until(conteudo_carregado)
    except TimeoutException:
        raise RuntimeError(
            "Conteúdo da página não carregou a tempo, ou nenhuma das "
            f"estações {ESTACOES_ALVO} apareceu no texto da página. "
            "Se o site mudou (ex: dados atrás de um botão/aba), pode ser "
            "necessário clicar em algo antes de ler o texto — veja o "
            "comentário sobre 'Consultar dados do gráfico' no topo do "
            "arquivo."
        )


def extrair_texto_renderizado(driver) -> str:
    """Pega o HTML já renderizado pelo navegador e devolve o texto puro
    da página, usando BeautifulSoup (é o bs4 quem faz a extração de texto,
    o Selenium só cuidou de executar o JavaScript)."""
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(" ", strip=True)


def extrair_leituras(texto: str) -> list[dict]:
    """Aplica a regex sobre o texto da página e retorna todas as leituras
    encontradas para as estações configuradas em ESTACOES_ALVO."""
    agora = datetime.now(TIMEZONE_BR) # Usa o fuso horário do Brasil
    registros = []

    for match in REGEX_LEITURA.finditer(texto):
        codigo, descricao, data_str, hora_str, nivel_str = match.groups()

        if codigo not in ESTACOES_ALVO:
            continue

        nivel_m = float(nivel_str.replace(",", "."))
        data_hora_medicao = datetime.strptime(
            f"{data_str} {hora_str}", "%d/%m/%Y %H:%M:%S"
        )

        registros.append({
            "estacao_codigo": codigo,
            "estacao_descricao": descricao.strip(),
            "data_hora_medicao": data_hora_medicao,
            "nivel_m": nivel_m,
            "capturado_em": agora,
        })

    if not registros:
        raise RuntimeError(
            f"Nenhuma leitura encontrada para as estações {ESTACOES_ALVO}. "
            "O layout/texto do site pode ter mudado - verifique a regex "
            "REGEX_LEITURA."
        )

    return registros


def salvar_historico(registros: list[dict]):
    """Agrupa os registros por estação e acrescenta ao Parquet histórico de
    cada uma, evitando duplicar a mesma medição (mesma estação + mesma
    data/hora de medição, já com segundos, informada pelo site)."""
    df_novos = pd.DataFrame(registros)

    # Obter a data da captura para criar a estrutura de diretórios
    # Assumimos que todos os registros no 'registros' têm o mesmo 'capturado_em'
    data_captura = df_novos['capturado_em'].iloc[0]

    # Criar o caminho dinâmico: OUTPUT_DIR / "parquet" / ano / mes / dia
    dynamic_output_dir = OUTPUT_DIR / "parquet" / str(data_captura.year) / f"{data_captura.month:02d}" / f"{data_captura.day:02d}"
    dynamic_output_dir.mkdir(parents=True, exist_ok=True)

    for codigo, grupo in df_novos.groupby("estacao_codigo"):
        # O arquivo será salvo dentro do diretório dinâmico
        arquivo = dynamic_output_dir / f"nivel_rio_{codigo.lower()}.parquet"

        if arquivo.exists():
            df_existente = pd.read_parquet(arquivo)
            combinado = pd.concat([df_existente, grupo], ignore_index=True)
            combinado = combinado.drop_duplicates(
                subset=["estacao_codigo", "data_hora_medicao"], keep="last"
            )
        else:
            combinado = grupo

        combinado = combinado.sort_values("data_hora_medicao").reset_index(drop=True)
        combinado.to_parquet(arquivo, index=False)

        ultima = combinado.iloc[-1]
        log.info(
            "[%s] histórico com %d linhas em %s | última leitura: %s m às %s",
            codigo,
            len(combinado),
            arquivo,
            ultima["nivel_m"],
            ultima["data_hora_medicao"],
        )


def main():
    driver = criar_driver()
    try:
        log.info("Abrindo %s ...", URL)
        driver.get(URL)
        esperar_conteudo_carregar(driver)
        texto = extrair_texto_renderizado(driver)
        registros = extrair_leituras(texto)
        salvar_historico(registros)
    finally:
        driver.quit()


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        log.error("Falha na captura: %s", e)
        sys.exit(1)

# LEITURA DE ARQUIVO UNICO

In [33]:
import pandas as pd

# Caminho completo para o arquivo Parquet
parquet_file_path = "/content/drive/MyDrive/Python/Analise Volume Rio/nivel_rio_dc06.parquet"

# Ler o arquivo Parquet em um DataFrame do pandas
try:
    df_historico = pd.read_parquet(parquet_file_path)
    print(f"Arquivo '{parquet_file_path}' lido com sucesso.")
    print("Primeiras 5 linhas do DataFrame:")
    display(df_historico)
except FileNotFoundError:
    print(f"Erro: O arquivo '{parquet_file_path}' não foi encontrado. Certifique-se de que o caminho está correto e o arquivo existe.")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo Parquet: {e}")

Erro: O arquivo '/content/drive/MyDrive/Python/Analise Volume Rio/nivel_rio_dc06.parquet' não foi encontrado. Certifique-se de que o caminho está correto e o arquivo existe.


# LEITURA TODOS PARQUETS

In [34]:
import pandas as pd
from pathlib import Path

# Define o diretório base onde os arquivos Parquet estão sendo salvos
base_parquet_dir = Path("/content/drive/MyDrive/Python/Analise Volume Rio") / "parquet"

# Lista para armazenar os DataFrames de cada arquivo Parquet
all_dfs = []

# Procura por todos os arquivos .parquet recursivamente dentro do diretório base
parquet_files = list(base_parquet_dir.glob("**/*.parquet"))

if not parquet_files:
    print(f"Nenhum arquivo Parquet encontrado em '{base_parquet_dir}'. Certifique-se de que o scraper já foi executado e criou os arquivos.")
else:
    print(f"Encontrados {len(parquet_files)} arquivos Parquet. Lendo e combinando...")
    for file_path in parquet_files:
        try:
            df = pd.read_parquet(file_path)
            all_dfs.append(df)
            print(f"Lido: {file_path.relative_to(base_parquet_dir)}")
        except Exception as e:
            print(f"Erro ao ler o arquivo {file_path}: {e}")

    if all_dfs:
        # Combina todos os DataFrames em um único DataFrame
        df_all_data = pd.concat(all_dfs, ignore_index=True)

        # Opcional: remover duplicatas se houver (útil se o scraper for executado várias vezes no mesmo dia/hora)
        df_all_data = df_all_data.drop_duplicates(subset=["estacao_codigo", "data_hora_medicao"], keep="last")
        df_all_data = df_all_data.sort_values(by=["estacao_codigo", "data_hora_medicao"]).reset_index(drop=True)

        print("\nDataFrame combinado criado com sucesso!")
        print(f"Total de linhas no DataFrame combinado: {len(df_all_data)}")
        print("Primeiras 5 linhas do DataFrame combinado:")
        display(df_all_data.tail(120))
        print("Informações do DataFrame combinado:")
        df_all_data.info()
    else:
        print("Nenhum dado foi lido com sucesso para criar o DataFrame combinado.")

Encontrados 5 arquivos Parquet. Lendo e combinando...
Lido: 2026/09/22/nivel_rio_dc06.parquet
Lido: 2026/09/23/nivel_rio_dc06.parquet
Lido: 2026/09/23/nivel_rio_dc03.parquet
Lido: 2026/09/23/nivel_rio_dc04.parquet
Lido: 2026/09/23/nivel_rio_dc05.parquet

DataFrame combinado criado com sucesso!
Total de linhas no DataFrame combinado: 598
Primeiras 5 linhas do DataFrame combinado:


,estacao_codigo,estacao_descricao,data_hora_medicao,nivel_m,capturado_em
478,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-22 22:30:55,1.16,2026-09-23 10:29:53.148395-03:00
479,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-22 22:40:56,1.20,2026-09-23 10:29:53.148395-03:00
480,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-22 22:50:56,1.21,2026-09-23 10:29:53.148395-03:00
481,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-22 23:00:55,1.20,2026-09-23 10:29:53.148395-03:00
482,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-22 23:10:55,1.20,2026-09-23 10:29:53.148395-03:00
...,...,...,...,...,...
593,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-23 16:50:56,1.15,2026-09-23 17:28:56.346129-03:00
594,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-23 17:01:06,1.12,2026-09-23 17:28:56.346129-03:00
595,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-23 17:11:31,1.10,2026-09-23 17:28:56.346129-03:00
596,DC06,Rio Itajaí-Mirim (curso antigo) - Itamirim Clu...,2026-09-23 17:20:42,0.35,2026-09-23 17:28:56.346129-03:00


Informações do DataFrame combinado:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype                            
---  ------             --------------  -----                            
 0   estacao_codigo     598 non-null    object                           
 1   estacao_descricao  598 non-null    object                           
 2   data_hora_medicao  598 non-null    datetime64[ns]                   
 3   nivel_m            598 non-null    float64                          
 4   capturado_em       598 non-null    datetime64[ns, America/Sao_Paulo]
dtypes: datetime64[ns, America/Sao_Paulo](1), datetime64[ns](1), float64(1), object(2)
memory usage: 23.5+ KB
